# Churn Prediction Mini-Project

**Dataset:** Telco Customer Churn (public dataset, Kaggle)
**Goal:** Clean data, engineer features, train a classifier, report accuracy + top predictive features.

> If `WA_Fn-UseC_-Telco-Customer-Churn.csv` (the real Kaggle file) is present in this folder, it is used automatically.
> Otherwise a synthetic dataset with the *same schema* is generated so the pipeline can be developed/tested end-to-end.

## Step 0: Imports

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

RANDOM_STATE = 42
DATA_PATH = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

## Step 1: Load Data

Download the real dataset from Kaggle ("Telco Customer Churn") and place the CSV next to this notebook to use real data. Otherwise a realistic synthetic dataset (same columns) is generated automatically.

In [10]:
def load_data(path=DATA_PATH, n_synthetic=2000):
    if os.path.exists(path):
        print(f"Loading REAL dataset from {path}")
        df = pd.read_csv(path)
        return df, False

    print("Real Kaggle CSV not found -> generating synthetic dataset with identical schema.")
    rng = np.random.default_rng(RANDOM_STATE)

    genders = rng.choice(["Male", "Female"], n_synthetic)
    senior = rng.choice([0, 1], n_synthetic, p=[0.84, 0.16])
    partner = rng.choice(["Yes", "No"], n_synthetic)
    dependents = rng.choice(["Yes", "No"], n_synthetic, p=[0.3, 0.7])
    tenure = rng.integers(0, 73, n_synthetic)
    phone_service = rng.choice(["Yes", "No"], n_synthetic, p=[0.9, 0.1])
    multiple_lines = rng.choice(["Yes", "No", "No phone service"], n_synthetic)
    internet_service = rng.choice(["DSL", "Fiber optic", "No"], n_synthetic, p=[0.35, 0.44, 0.21])
    contract = rng.choice(["Month-to-month", "One year", "Two year"], n_synthetic, p=[0.55, 0.21, 0.24])
    paperless = rng.choice(["Yes", "No"], n_synthetic, p=[0.59, 0.41])
    payment_method = rng.choice(
        ["Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"],
        n_synthetic
    )
    monthly_charges = np.round(rng.uniform(18, 120, n_synthetic), 2)
    total_charges = np.round(monthly_charges * tenure + rng.normal(0, 50, n_synthetic), 2)
    total_charges = np.clip(total_charges, 0, None)

    online_security = rng.choice(["Yes", "No", "No internet service"], n_synthetic)
    tech_support = rng.choice(["Yes", "No", "No internet service"], n_synthetic)
    streaming_tv = rng.choice(["Yes", "No", "No internet service"], n_synthetic)
    streaming_movies = rng.choice(["Yes", "No", "No internet service"], n_synthetic)
    online_backup = rng.choice(["Yes", "No", "No internet service"], n_synthetic)
    device_protection = rng.choice(["Yes", "No", "No internet service"], n_synthetic)

    churn_score = (
        (contract == "Month-to-month") * 0.35
        + (tenure < 12) * 0.25
        + (monthly_charges > 80) * 0.15
        + rng.normal(0, 0.15, n_synthetic)
    )
    churn_prob = 1 / (1 + np.exp(-(churn_score - 0.3) * 4))
    churn = (rng.uniform(0, 1, n_synthetic) < churn_prob)
    churn = np.where(churn, "Yes", "No")

    df = pd.DataFrame({
        "customerID": [f"CUST-{i:05d}" for i in range(n_synthetic)],
        "gender": genders, "SeniorCitizen": senior, "Partner": partner,
        "Dependents": dependents, "tenure": tenure, "PhoneService": phone_service,
        "MultipleLines": multiple_lines, "InternetService": internet_service,
        "OnlineSecurity": online_security, "OnlineBackup": online_backup,
        "DeviceProtection": device_protection, "TechSupport": tech_support,
        "StreamingTV": streaming_tv, "StreamingMovies": streaming_movies,
        "Contract": contract, "PaperlessBilling": paperless,
        "PaymentMethod": payment_method, "MonthlyCharges": monthly_charges,
        "TotalCharges": total_charges, "Churn": churn,
    })
    return df, True

df_raw, is_synthetic = load_data()
print(f"Dataset shape: {df_raw.shape}")
df_raw.head()

Loading REAL dataset from WA_Fn-UseC_-Telco-Customer-Churn.csv
Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Step 2: Clean Data

- Convert `TotalCharges` to numeric (real dataset has blank strings for new customers)
- Fill missing values with the median
- Drop the ID column (not predictive)
- Drop duplicate rows

In [3]:
def clean_data(df):
    df = df.copy()
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())
    if "customerID" in df.columns:
        df = df.drop(columns=["customerID"])
    df = df.drop_duplicates()
    return df

df_clean = clean_data(df_raw)
df_clean.isna().sum().sum(), df_clean.shape

(np.int64(0), (7021, 20))

## Step 3: Feature Engineering

Three new features:
1. **tenure_group** — bucket customers into tenure ranges (new customers churn more)
2. **avg_monthly_spend** — total charges / tenure (spend rate signal)
3. **num_addon_services** — count of add-on services subscribed (engagement signal)

In [4]:
def engineer_features(df):
    df = df.copy()
    df["tenure_group"] = pd.cut(
        df["tenure"], bins=[-1, 12, 24, 48, 72],
        labels=["0-1yr", "1-2yr", "2-4yr", "4-6yr"]
    )
    df["avg_monthly_spend"] = df["TotalCharges"] / df["tenure"].replace(0, 1)
    addon_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                  "TechSupport", "StreamingTV", "StreamingMovies"]
    df["num_addon_services"] = (df[addon_cols] == "Yes").sum(axis=1)
    return df

df_features = engineer_features(df_clean)
df_features[["tenure_group", "avg_monthly_spend", "num_addon_services"]].head()

,tenure_group,avg_monthly_spend,num_addon_services
0,0-1yr,29.850000,1
1,2-4yr,55.573529,2
2,0-1yr,54.075000,2
3,2-4yr,40.905556,3
4,0-1yr,75.825000,0


## Step 4: Encode Categorical Columns

In [5]:
def prepare_model_data(df):
    df = df.copy()
    target = df["Churn"].map({"Yes": 1, "No": 0})
    features = df.drop(columns=["Churn"])
    encoders = {}
    for col in features.columns:
        if str(features[col].dtype) in ("object", "category", "str", "string"):
            le = LabelEncoder()
            features[col] = le.fit_transform(features[col].astype(str))
            encoders[col] = le
    return features, target, encoders

X, y, encoders = prepare_model_data(df_features)
X.shape, y.value_counts(normalize=True)

((7021, 22),
 Churn
 0    0.735508
 1    0.264492
 Name: proportion, dtype: float64)

## Step 5: Train Models

Logistic Regression and Decision Tree, 80/20 train-test split, stratified on the target.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Logistic Regression (scaled features)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
log_reg.fit(X_train_scaled, y_train)
log_preds = log_reg.predict(X_test_scaled)
log_acc = accuracy_score(y_test, log_preds)

# Decision Tree
tree = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)
tree.fit(X_train, y_train)
tree_preds = tree.predict(X_test)
tree_acc = accuracy_score(y_test, tree_preds)

print(f"Logistic Regression accuracy: {log_acc:.4f}")
print(f"Decision Tree accuracy:       {tree_acc:.4f}")

Logistic Regression accuracy: 0.8000
Decision Tree accuracy:       0.7886


## Step 6: Top 3 Predictive Features

In [7]:
def top_features(model, feature_names, model_type, n=3):
    if model_type == "logreg":
        importance = np.abs(model.coef_[0])
    else:
        importance = model.feature_importances_
    idx = np.argsort(importance)[::-1][:n]
    return [(feature_names[i], round(importance[i], 4)) for i in idx]

feature_names = list(X_train.columns)
print("Logistic Regression top 3:", top_features(log_reg, feature_names, "logreg"))
print("Decision Tree top 3:      ", top_features(tree, feature_names, "tree"))

Logistic Regression top 3: [('tenure', np.float64(1.2482)), ('num_addon_services', np.float64(0.725)), ('TotalCharges', np.float64(0.6183))]
Decision Tree top 3:       [('Contract', np.float64(0.5163)), ('tenure', np.float64(0.1461)), ('OnlineSecurity', np.float64(0.1302))]


## Step 7: Summary

- **Data source:** replace `WA_Fn-UseC_-Telco-Customer-Churn.csv` with the real Kaggle file for production-quality numbers; results above run on synthetic data if the real file isn't present.
- **Classification report** below gives precision/recall/F1 per class.

In [8]:
print("Logistic Regression report:\n", classification_report(y_test, log_preds))
print("Decision Tree report:\n", classification_report(y_test, tree_preds))

Logistic Regression report:
               precision    recall  f1-score   support

           0       0.84      0.90      0.87      1033
           1       0.65      0.53      0.58       372

    accuracy                           0.80      1405
   macro avg       0.75      0.71      0.73      1405
weighted avg       0.79      0.80      0.79      1405

Decision Tree report:
               precision    recall  f1-score   support

           0       0.83      0.90      0.86      1033
           1       0.64      0.47      0.54       372

    accuracy                           0.79      1405
   macro avg       0.73      0.69      0.70      1405
weighted avg       0.78      0.79      0.78      1405

